# CardIA: Clusterización de tarjetas de crédito

## Objetivo

El objetivo de este notebook es construir un modelo **no supervisado** que agrupe tarjetas de crédito con características similares. La unidad de análisis será una tarjeta identificada por `ID_Tarjeta` y se integrarán cuatro fuentes del archivo `tarjetas.xlsx`:

- **Tarjetas:** características financieras generales del producto.
- **Requisitos:** condiciones mínimas solicitadas al cliente.
- **Comisiones:** costos y penalizaciones asociados al uso de la tarjeta.
- **Beneficios:** recompensas, meses sin intereses, descuentos, seguros, preventas y transferencias de saldo.

La intención no es formar grupos a partir del nombre de la institución o de etiquetas comerciales previamente definidas, sino permitir que los grupos surjan de las características financieras, requisitos, costos y beneficios de cada producto.

El flujo metodológico conserva la lógica utilizada previamente en un ejercicio de clustering: **EDA → ingeniería de variables → preparación y escalamiento → PCA → K-Means → clustering aglomerativo → comparación → visualización → perfilamiento**.

## Decisiones metodológicas

Antes de comenzar se establecen las siguientes decisiones:

1. **No se eliminarán valores atípicos.** La base contiene pocas tarjetas y un producto extremo puede representar un segmento real del mercado —por ejemplo, tarjetas con líneas de crédito, anualidades o beneficios muy superiores al resto—. Los atípicos se observarán durante el EDA, pero no serán descartados automáticamente.
2. **Los datos faltantes de requisitos no representan errores de captura.** Un valor faltante significa que la institución no exige ese requisito. Por ello no se utilizarán métodos de imputación estadística. En variables que representan mínimos exigidos, el valor faltante se codificará como cero cuando cero tenga la interpretación natural de “sin requisito”, acompañado cuando sea útil por una marca de presencia.
3. **Los beneficios no se reducirán únicamente a variables binarias.** Se extraerán tasas, plazos, puntos, número de categorías cubiertas, número de modalidades y otras medidas numéricas disponibles en los textos.
4. **Los montos y porcentajes de comisiones se conservarán por separado.** No es correcto sumar pesos, dólares y porcentajes como si fueran la misma unidad.
5. **`ID_Tarjeta`, nombre, institución y clase se conservarán para identificación y perfilamiento.** Por defecto no entrarán al clustering. Se incluyen interruptores para probar `Clase` o `Institución` mediante One-Hot Encoding únicamente si posteriormente se considera metodológicamente conveniente.
6. Se compararán cuatro escenarios: **K-Means sin PCA, K-Means con PCA al 90%, aglomerativo sin PCA y aglomerativo con PCA al 90%**.
7. El PCA de 2 y 3 componentes se utilizará para visualización. El PCA utilizado para modelar conservará aproximadamente el **90% de la varianza explicada**.

## Librerías

Se utilizan principalmente `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn` y `scipy`, manteniendo herramientas estándar de preparación, reducción de dimensionalidad y clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import re
import unicodedata
import warnings

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import linkage, dendrogram

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:,.3f}".format

RANDOM_STATE = 42

## Funciones auxiliares

Estas funciones se emplearán para mantener el notebook ordenado. Se incluyen herramientas para completitud, descripción estadística, normalización de texto, visualización de variables y evaluación de clustering.

In [ ]:
def desc_table(datos, vars_, tipo="num"):
    """Tabla descriptiva semejante a la utilizada en el ejercicio previo."""
    if tipo == "num":
        x = datos[vars_].describe(
            percentiles=(.01, .05, .25, .30, .40, .50, .75, .90, .95, .98, .99)
        ).T.reset_index()
    else:
        x = datos[vars_].describe(include="all").T.reset_index()
    x["%nulos"] = 1 - (x["count"] / len(datos))
    return x


def tabla_completitud(df):
    return pd.DataFrame({
        "variable": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "nulos": df.isna().sum().values,
        "% nulos": (df.isna().mean().values * 100).round(2),
        "unicos": df.nunique(dropna=True).values
    }).sort_values(["% nulos", "variable"], ascending=[False, True]).reset_index(drop=True)


def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower().strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto)
    return texto


def slug(texto):
    texto = normalizar_texto(texto)
    texto = re.sub(r"[^a-z0-9]+", "_", texto).strip("_")
    return texto


def graficar_numericas(df, columnas, ncols=3, bins=20):
    columnas = [c for c in columnas if c in df.columns]
    if not columnas:
        return
    nrows = int(np.ceil(len(columnas) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.6*nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, columnas):
        sb.histplot(df[col].dropna(), kde=True, bins=bins, ax=ax)
        ax.set_title(col)
    for ax in axes[len(columnas):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def pares_correlacion_alta(df, umbral=0.85):
    corr = df.corr(numeric_only=True)
    mascara = np.triu(np.ones(corr.shape), k=1).astype(bool)
    pares = (
        corr.where(mascara)
        .stack()
        .rename("correlacion")
        .reset_index()
        .rename(columns={"level_0": "variable_1", "level_1": "variable_2"})
    )
    pares["abs_correlacion"] = pares["correlacion"].abs()
    return pares[pares["abs_correlacion"] >= umbral].sort_values(
        "abs_correlacion", ascending=False
    )

In [ ]:
def evaluar_kmeans(X, k_min=2, k_max=10):
    resultados = []
    k_max = min(k_max, len(X) - 1)
    for k in range(k_min, k_max + 1):
        modelo = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = modelo.fit_predict(X)
        resultados.append({
            "k": k,
            "inercia": modelo.inertia_,
            "silhouette": silhouette_score(X, labels)
        })
    return pd.DataFrame(resultados)


def graficar_codo_silueta(resultados, titulo="K-Means"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(resultados["k"], resultados["inercia"], marker="o")
    axes[0].set_title(f"Método del codo - {titulo}")
    axes[0].set_xlabel("Número de clusters (k)")
    axes[0].set_ylabel("Inercia")

    axes[1].plot(resultados["k"], resultados["silhouette"], marker="o")
    axes[1].set_title(f"Silueta promedio - {titulo}")
    axes[1].set_xlabel("Número de clusters (k)")
    axes[1].set_ylabel("Silhouette score")
    plt.tight_layout()
    plt.show()


def grafica_silueta(X, labels, titulo="Gráfica de silueta"):
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)
    y_lower = 10
    plt.figure(figsize=(9, 6))
    for cluster in sorted(np.unique(labels)):
        vals_cluster = np.sort(valores[labels == cluster])
        size = len(vals_cluster)
        y_upper = y_lower + size
        plt.fill_betweenx(np.arange(y_lower, y_upper), 0, vals_cluster, alpha=0.7)
        plt.text(-0.05, y_lower + 0.5 * size, str(cluster))
        y_lower = y_upper + 10
    plt.axvline(promedio, linestyle="--", label=f"Silueta promedio = {promedio:.3f}")
    plt.xlabel("Coeficiente de silueta")
    plt.ylabel("Cluster")
    plt.title(titulo)
    plt.legend()
    plt.show()


def evaluar_aglomerativo(X, k_min=2, k_max=10, linkage_method="ward"):
    resultados = []
    k_max = min(k_max, len(X) - 1)
    for k in range(k_min, k_max + 1):
        modelo = AgglomerativeClustering(n_clusters=k, linkage=linkage_method)
        labels = modelo.fit_predict(X)
        resultados.append({
            "k": k,
            "silhouette": silhouette_score(X, labels)
        })
    return pd.DataFrame(resultados)


def graficar_silueta_aglomerativo(resultados, titulo="Aglomerativo"):
    plt.figure(figsize=(7, 4))
    plt.plot(resultados["k"], resultados["silhouette"], marker="o")
    plt.xlabel("Número de clusters (k)")
    plt.ylabel("Silhouette score")
    plt.title(f"Silueta promedio - {titulo}")
    plt.show()


def graficar_dendrograma(X, titulo="Dendrograma", metodo="ward"):
    Z = linkage(X, method=metodo, metric="euclidean")
    plt.figure(figsize=(16, 7))
    dendrogram(Z)
    plt.title(titulo)
    plt.xlabel("Observaciones")
    plt.ylabel("Distancia")
    plt.show()
    return Z

# Datos

## Carga del archivo

En Google Colab se puede subir `tarjetas.xlsx` al panel de archivos y conservar la siguiente ruta. Si el nombre cambia, únicamente debe modificarse `RUTA_ARCHIVO`.

In [ ]:
RUTA_ARCHIVO = "/content/tarjetas.xlsx"

hojas = pd.ExcelFile(RUTA_ARCHIVO).sheet_names
print("Hojas disponibles:", hojas)

# dtype=str en el identificador evita perder ceros a la izquierda.
tarjetas = pd.read_excel(RUTA_ARCHIVO, sheet_name="Tarjetas", dtype={"ID_Tarjeta": str})
requisitos = pd.read_excel(RUTA_ARCHIVO, sheet_name="Requisitos", dtype={"ID_Tarjeta": str})
comisiones = pd.read_excel(RUTA_ARCHIVO, sheet_name="Comisiones", dtype={"ID_Tarjeta": str})
beneficios = pd.read_excel(RUTA_ARCHIVO, sheet_name="Beneficios", dtype={"ID_Tarjeta": str})

for df in [tarjetas, requisitos, comisiones, beneficios]:
    df["ID_Tarjeta"] = df["ID_Tarjeta"].astype(str).str.zfill(3)

print("Tarjetas:", tarjetas.shape)
print("Requisitos:", requisitos.shape)
print("Comisiones:", comisiones.shape)
print("Beneficios:", beneficios.shape)

## Validación de integridad

Antes de realizar ingeniería de variables se verifica que `ID_Tarjeta` funcione correctamente como llave. `Tarjetas` y `Requisitos` deberían tener una fila por tarjeta; `Comisiones` y `Beneficios` pueden tener varias filas por tarjeta porque representan relaciones uno-a-muchos.

In [ ]:
print("Duplicados por ID en Tarjetas:", tarjetas["ID_Tarjeta"].duplicated().sum())
print("Duplicados por ID en Requisitos:", requisitos["ID_Tarjeta"].duplicated().sum())

ids_tarjetas = set(tarjetas["ID_Tarjeta"])
for nombre, df in {
    "Requisitos": requisitos,
    "Comisiones": comisiones,
    "Beneficios": beneficios
}.items():
    ids = set(df["ID_Tarjeta"])
    print(f"\n{nombre}")
    print("IDs presentes:", len(ids))
    print("IDs que faltan respecto a Tarjetas:", sorted(ids_tarjetas - ids))
    print("IDs adicionales:", sorted(ids - ids_tarjetas))

# Exploración y limpieza de datos (EDA)

El EDA se realiza primero sobre las tablas originales. El objetivo es entender las escalas, distribución, completitud y cardinalidad antes de convertir la información a una matriz de una fila por tarjeta.

## Tarjetas

Esta tabla contiene las variables financieras generales del producto. `ID_Tarjeta`, nombre e institución sirven para identificar la tarjeta; `Clase` se conservará para perfilamiento y podrá incorporarse opcionalmente al modelo.

In [ ]:
tarjetas.info()
display(tarjetas.head())
display(tabla_completitud(tarjetas))

In [ ]:
vars_tarjetas_num = [
    "CAT\npublicidad\n(%)",
    "Anualidad\n($)*",
    "Tasa de\ninteres de\ncontrato\n(%)",
    "Línea de\ncrédito\ndesde ($)",
    "Conteo requisitos",
    "Conteo comisiones",
    "Conteo beneficios"
]
display(desc_table(tarjetas, vars_tarjetas_num, "num"))
graficar_numericas(tarjetas, vars_tarjetas_num)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
tarjetas["Institución"].value_counts().plot(kind="bar", ax=axes[0], title="Tarjetas por institución")
tarjetas["Clase"].value_counts().plot(kind="bar", ax=axes[1], title="Tarjetas por clase")
plt.tight_layout()
plt.show()

## Requisitos

En esta fuente, un valor faltante **no se tratará como dato perdido**. Su interpretación es que ese requisito no se exige. Por ello, la completitud se estudia para conocer la política de requisitos del mercado, no para decidir un método de imputación.

In [ ]:
requisitos.info()
display(requisitos.head())
display(tabla_completitud(requisitos))

vars_req_num = [c for c in requisitos.columns if c not in ["Tarjeta", "ID_Tarjeta"]]
display(desc_table(requisitos, vars_req_num, "num"))
graficar_numericas(requisitos, vars_req_num)

## Comisiones

Las comisiones tienen tres elementos relevantes: **concepto**, **monto** y **denominación**. Un mismo valor numérico no tiene la misma interpretación si se expresa en pesos, dólares o como porcentaje del monto de una transacción. Por esta razón las denominaciones se mantendrán separadas durante la ingeniería de variables.

In [ ]:
comisiones.info()
display(comisiones.head(10))
display(tabla_completitud(comisiones))

print("Conceptos de comisión:")
display(comisiones["Concepto"].value_counts().to_frame("registros"))
print("\nDenominaciones:")
display(comisiones["Denominación"].value_counts().to_frame("registros"))
print("\nTipos:")
display(comisiones["Tipo"].value_counts().to_frame("registros"))

In [ ]:
display(desc_table(comisiones, ["Monto"], "num"))

plt.figure(figsize=(9, 4))
sb.boxplot(data=comisiones, x="Denominación", y="Monto")
plt.yscale("symlog")
plt.title("Distribución de montos de comisión por denominación")
plt.show()

## Beneficios

Esta tabla es la principal fuente de ingeniería de variables. Los textos contienen información que puede representarse cuantitativamente sin utilizar NLP avanzado: puntos por unidad de gasto, tasas de transferencia, plazos, número de categorías cubiertas por MSI o descuentos y amplitud de coberturas de seguros o preventas.

In [ ]:
beneficios.info()
display(beneficios.head(15))
display(tabla_completitud(beneficios))

display(beneficios["Tipo"].value_counts().to_frame("registros"))

In [ ]:
for tipo, bloque in beneficios.groupby("Tipo"):
    print("\n", "="*80)
    print(tipo)
    display(bloque[["Tarjeta", "Texto"]].head(8))

# Exploración e ingeniería de variables

La meta de esta sección es transformar las cuatro tablas a **una sola fila por `ID_Tarjeta`**. Se priorizan variables interpretables, con unidades claras y útiles para distinguir productos.

## Ingeniería de variables de Tarjetas

Se renombran las columnas principales y se conservan cuatro variables financieras continuas como núcleo del producto:

- CAT publicado.
- Anualidad.
- Tasa de interés contractual.
- Línea de crédito inicial.

Los conteos originales se conservarán únicamente para revisión, ya que posteriormente se construirán medidas más detalladas a partir de `Comisiones` y `Beneficios`.

In [ ]:
tarjetas_fe = tarjetas.rename(columns={
    "Nombre de la tarjeta": "nombre_tarjeta",
    "Institución": "institucion",
    "CAT\npublicidad\n(%)": "cat_pct",
    "Anualidad\n($)*": "anualidad_pesos",
    "Tasa de\ninteres de\ncontrato\n(%)": "tasa_interes_pct",
    "Línea de\ncrédito\ndesde ($)": "linea_credito_desde",
    "Clase": "clase",
    "Duplicado": "duplicado",
    "Conteo requisitos": "conteo_requisitos_original",
    "Conteo comisiones": "conteo_comisiones_original",
    "Conteo beneficios": "conteo_beneficios_original"
}).copy()

tarjetas_fe = tarjetas_fe.sort_values("ID_Tarjeta").reset_index(drop=True)
display(tarjetas_fe.head())

## Ingeniería de variables de Requisitos

Para los requisitos se aprovecha el significado de los faltantes:

- Score mínimo faltante → no se exige score mínimo → `score_min_modelo = 0`.
- Antigüedad laboral/residencial faltante → no se exige → valor estructural 0.
- Ingreso mínimo faltante → no se exige → valor estructural 0.
- Edad máxima faltante significa **sin límite superior reportado**. No se reemplaza directamente por 0, porque eso tendría una interpretación incorrecta. En su lugar se construye `restriccion_edad_maxima`: cero significa que no existe restricción superior; valores mayores representan un límite más restrictivo.

También se agregan indicadores de presencia de requisito. No sustituyen a los valores numéricos: los complementan.

In [ ]:
requisitos_fe = requisitos.rename(columns={
    "Tarjeta": "tarjeta_req",
    "Edad mínima": "edad_minima",
    "Edad máxima": "edad_maxima",
    "Score de crédito mínimo": "score_minimo",
    "Antigüedad laboral mínima": "antiguedad_laboral_minima",
    "Antigüedad residencial mínima": "antiguedad_residencial_minima",
    "Ingreso mensual mínimo": "ingreso_mensual_minimo"
}).copy()

# Indicadores: permiten distinguir "no se exige" de un valor positivo.
for col in [
    "score_minimo",
    "antiguedad_laboral_minima",
    "antiguedad_residencial_minima",
    "ingreso_mensual_minimo"
]:
    requisitos_fe[f"requiere_{col}"] = requisitos_fe[col].notna().astype(int)
    requisitos_fe[f"{col}_modelo"] = requisitos_fe[col].fillna(0)

# Edad máxima: 0 en esta nueva variable representa ausencia de restricción superior.
EDAD_TOPE_OPERATIVA = 100
requisitos_fe["requiere_edad_maxima"] = requisitos_fe["edad_maxima"].notna().astype(int)
requisitos_fe["restriccion_edad_maxima"] = np.where(
    requisitos_fe["edad_maxima"].notna(),
    EDAD_TOPE_OPERATIVA - requisitos_fe["edad_maxima"],
    0
)

requisitos_modelo = requisitos_fe[[
    "ID_Tarjeta",
    "edad_minima",
    "score_minimo_modelo",
    "antiguedad_laboral_minima_modelo",
    "antiguedad_residencial_minima_modelo",
    "ingreso_mensual_minimo_modelo",
    "restriccion_edad_maxima",
    "requiere_score_minimo",
    "requiere_antiguedad_laboral_minima",
    "requiere_antiguedad_residencial_minima",
    "requiere_ingreso_mensual_minimo",
    "requiere_edad_maxima"
]].copy()

display(requisitos_modelo.head())

## Ingeniería de variables de Comisiones

Se construyen dos niveles de variables:

1. **Variables específicas por concepto y denominación**, por ejemplo una comisión de anualidad en pesos o un porcentaje por disposición de efectivo. Esto conserva el detalle del producto.
2. **Variables resumen**, como número de conceptos cobrados, suma/promedio/máximo de comisiones en cada unidad y cargas por tipo (`Obligatoria`, `Penalización`, `Por evento`).

No se combinan pesos, dólares y porcentajes en una misma magnitud.

In [ ]:
comisiones_fe = comisiones.copy()
comisiones_fe["concepto_norm"] = comisiones_fe["Concepto"].map(slug)
comisiones_fe["tipo_norm"] = comisiones_fe["Tipo"].map(slug)

# Normalización de la unidad de cada comisión.
def normalizar_denominacion(x):
    t = normalizar_texto(x)
    if "peso" in t:
        return "pesos"
    if "dolar" in t:
        return "dolares"
    if "monto de la transaccion" in t:
        return "pct_transaccion"
    return slug(t) if t else "sin_unidad"

comisiones_fe["denom_norm"] = comisiones_fe["Denominación"].map(normalizar_denominacion)
comisiones_fe["Monto"] = pd.to_numeric(comisiones_fe["Monto"], errors="coerce")

display(comisiones_fe[["Concepto", "Monto", "Denominación", "Tipo", "concepto_norm", "denom_norm"]].head(15))

In [ ]:
# 1) Pivot de montos específicos por concepto y denominación.
pivot_comisiones = comisiones_fe.pivot_table(
    index="ID_Tarjeta",
    columns=["concepto_norm", "denom_norm"],
    values="Monto",
    aggfunc="max",
    fill_value=0
)

pivot_comisiones.columns = [
    f"com_{concepto}_{unidad}" for concepto, unidad in pivot_comisiones.columns
]
pivot_comisiones = pivot_comisiones.reset_index()

# 2) Agregados por denominación.
agg_den = (
    comisiones_fe
    .groupby(["ID_Tarjeta", "denom_norm"])["Monto"]
    .agg(["count", "sum", "mean", "max"])
    .reset_index()
)

partes = []
for estadistico in ["count", "sum", "mean", "max"]:
    tmp = agg_den.pivot(index="ID_Tarjeta", columns="denom_norm", values=estadistico).fillna(0)
    tmp.columns = [f"com_{estadistico}_{slug(c)}" for c in tmp.columns]
    partes.append(tmp)

resumen_den = pd.concat(partes, axis=1).reset_index()

# 3) Agregados por tipo de comisión y unidad.
agg_tipo = (
    comisiones_fe
    .groupby(["ID_Tarjeta", "tipo_norm", "denom_norm"])["Monto"]
    .agg(["count", "sum", "max"])
    .reset_index()
)

partes_tipo = []
for estadistico in ["count", "sum", "max"]:
    tmp = agg_tipo.pivot_table(
        index="ID_Tarjeta",
        columns=["tipo_norm", "denom_norm"],
        values=estadistico,
        fill_value=0
    )
    tmp.columns = [f"com_{estadistico}_{tipo}_{unidad}" for tipo, unidad in tmp.columns]
    partes_tipo.append(tmp)

resumen_tipo = pd.concat(partes_tipo, axis=1).reset_index()

# 4) Diversidad de comisiones.
diversidad_com = comisiones_fe.groupby("ID_Tarjeta").agg(
    com_n_registros=("Concepto", "size"),
    com_n_conceptos=("Concepto", "nunique"),
    com_n_tipos=("Tipo", "nunique")
).reset_index()

comisiones_modelo = (
    pivot_comisiones
    .merge(resumen_den, on="ID_Tarjeta", how="outer")
    .merge(resumen_tipo, on="ID_Tarjeta", how="outer")
    .merge(diversidad_com, on="ID_Tarjeta", how="outer")
    .fillna(0)
)

print("Dimensión de variables de comisiones:", comisiones_modelo.shape)
display(comisiones_modelo.head())

## Ingeniería de variables de Beneficios

Esta sección busca conservar la mayor cantidad posible de información **numérica e interpretable**.

La estrategia es diferente según el tipo de beneficio:

- **Puntos:** cantidad de puntos por unidad de gasto, separando pesos y dólares.
- **Transferencia de saldo:** tasa mínima/promedio/máxima y plazos mínimo/máximo; una tarjeta puede tener más de una alternativa.
- **MSI y descuentos:** número de categorías cubiertas, cantidad de registros/modalidades y presencia de cada categoría.
- **Preventas:** amplitud de eventos cubiertos.
- **Seguros:** número y tipo de coberturas.
- **Anualidad:** identificación del beneficio “sin anualidad”.
- **Resumen general:** cantidad de registros, tipos de beneficio y amplitud total.

Los indicadores binarios se conservan cuando son útiles, pero el objetivo es que estén acompañados por medidas de intensidad o amplitud.

In [ ]:
beneficios_fe = beneficios.copy()
beneficios_fe["texto_norm"] = beneficios_fe["Texto"].map(normalizar_texto)
beneficios_fe["tipo_norm"] = beneficios_fe["Tipo"].map(slug)

display(beneficios_fe[["ID_Tarjeta", "Tipo", "Texto", "texto_norm"]].head(10))

### Puntos

Se extrae el número de puntos ofrecidos y la unidad de gasto mencionada. No se convierte dólar a peso porque eso requeriría introducir un tipo de cambio externo y variable; ambos esquemas se conservan separados.

In [ ]:
puntos = beneficios_fe[beneficios_fe["Tipo"].eq("Puntos")].copy()

puntos["puntos_por_unidad"] = pd.to_numeric(
    puntos["texto_norm"].str.extract(r"(\d+(?:[\.,]\d+)?)\s+puntos?")[0].str.replace(",", ".", regex=False),
    errors="coerce"
)
puntos["es_por_peso"] = puntos["texto_norm"].str.contains(r"cada peso", regex=True).astype(int)
puntos["es_por_dolar"] = puntos["texto_norm"].str.contains(r"cada dolar", regex=True).astype(int)
puntos["puntos_por_peso"] = np.where(puntos["es_por_peso"].eq(1), puntos["puntos_por_unidad"], 0)
puntos["puntos_por_dolar"] = np.where(puntos["es_por_dolar"].eq(1), puntos["puntos_por_unidad"], 0)

puntos_modelo = puntos.groupby("ID_Tarjeta").agg(
    benef_puntos_n_esquemas=("Tipo", "size"),
    benef_puntos_max=("puntos_por_unidad", "max"),
    benef_puntos_promedio=("puntos_por_unidad", "mean"),
    benef_puntos_por_peso_max=("puntos_por_peso", "max"),
    benef_puntos_por_dolar_max=("puntos_por_dolar", "max")
).reset_index()

display(puntos_modelo.head())

### Transferencia de saldo

Algunas tarjetas ofrecen varias alternativas de transferencia. En vez de conservar una sola, se resumen la mejor y peor tasa observada, la tasa promedio y el rango de plazos disponibles.

In [ ]:
transfer = beneficios_fe[beneficios_fe["Tipo"].eq("Transferencia de Saldo")].copy()

transfer["tasa_transferencia_pct"] = pd.to_numeric(
    transfer["texto_norm"].str.extract(r"tasa del\s*(\d+(?:[\.,]\d+)?)\s*%")[0].str.replace(",", ".", regex=False),
    errors="coerce"
)

plazos = transfer["texto_norm"].str.extract(
    r"entre\s*(\d+)\s*y\s*(\d+)\s*meses"
)
transfer["plazo_min_meses"] = pd.to_numeric(plazos[0], errors="coerce")
transfer["plazo_max_meses"] = pd.to_numeric(plazos[1], errors="coerce")

transfer_modelo = transfer.groupby("ID_Tarjeta").agg(
    benef_transfer_n_opciones=("Tipo", "size"),
    benef_transfer_tasa_min=("tasa_transferencia_pct", "min"),
    benef_transfer_tasa_promedio=("tasa_transferencia_pct", "mean"),
    benef_transfer_tasa_max=("tasa_transferencia_pct", "max"),
    benef_transfer_plazo_min=("plazo_min_meses", "min"),
    benef_transfer_plazo_max=("plazo_max_meses", "max")
).reset_index()

display(transfer_modelo.head())

### MSI y descuentos: amplitud de categorías

Para beneficios expresados como listas de establecimientos o servicios, la intensidad se aproxima mediante el **número de categorías cubiertas**. Además se conservan indicadores por categoría para no perder la naturaleza del beneficio.

In [ ]:
CATEGORIAS_CONSUMO_VIAJE = {
    "autoservicio_departamental": r"autoservicio|departamental",
    "salud": r"salud",
    "restaurantes": r"restaurante",
    "entretenimiento": r"entretenimiento",
    "educativos": r"educativ",
    "comercio_en_linea": r"comercios? en linea|comercio en linea",
    "agencias_viajes": r"agencias? de viajes?",
    "peaje": r"peaje",
    "renta_auto": r"renta de auto",
    "transporte_aereo": r"transporte aereo",
    "transporte_terrestre": r"transporte terrestre",
    "hoteles": r"hoteles?",
    "otros": r"\botros\b"
}

CATEGORIAS_VIAJE = {
    "agencias_viajes", "peaje", "renta_auto", "transporte_aereo",
    "transporte_terrestre", "hoteles"
}


def construir_features_categorias(df_beneficios, tipo, prefijo, categorias, categorias_viaje=None):
    tmp = df_beneficios[df_beneficios["Tipo"].eq(tipo)].copy()
    if tmp.empty:
        return pd.DataFrame(columns=["ID_Tarjeta"])

    cols_flags = []
    for nombre, patron in categorias.items():
        col = f"{prefijo}_{nombre}"
        tmp[col] = tmp["texto_norm"].str.contains(patron, regex=True).astype(int)
        cols_flags.append(col)

    flags = tmp.groupby("ID_Tarjeta")[cols_flags].max()
    resumen = tmp.groupby("ID_Tarjeta").size().rename(f"{prefijo}_n_registros").to_frame()
    salida = resumen.join(flags)
    salida[f"{prefijo}_n_categorias"] = salida[cols_flags].sum(axis=1)

    if categorias_viaje is None:
        categorias_viaje = set()
    viaje_cols = [f"{prefijo}_{c}" for c in categorias if c in categorias_viaje]
    consumo_cols = [c for c in cols_flags if c not in viaje_cols and not c.endswith("_otros")]
    salida[f"{prefijo}_n_categorias_viaje"] = salida[viaje_cols].sum(axis=1)
    salida[f"{prefijo}_n_categorias_consumo"] = salida[consumo_cols].sum(axis=1)
    return salida.reset_index()


msi_modelo = construir_features_categorias(
    beneficios_fe, "Meses sin intereses", "benef_msi", CATEGORIAS_CONSUMO_VIAJE, CATEGORIAS_VIAJE
)

descuentos_modelo = construir_features_categorias(
    beneficios_fe, "Descuentos", "benef_desc", CATEGORIAS_CONSUMO_VIAJE, CATEGORIAS_VIAJE
)

display(msi_modelo.head())
display(descuentos_modelo.head())

### Preventas

Se mide tanto la existencia del beneficio como su amplitud: conciertos/festivales, teatro, eventos deportivos y otros eventos.

In [ ]:
CATEGORIAS_PREVENTA = {
    "conciertos_festivales": r"conciertos?|festivales?",
    "teatro": r"teatro",
    "eventos_deportivos": r"eventos? deportivos?",
    "otros": r"\botros\b"
}

preventas_modelo = construir_features_categorias(
    beneficios_fe, "Preventas", "benef_preventa", CATEGORIAS_PREVENTA
)

display(preventas_modelo.head())

### Seguros

Los seguros se descomponen por cobertura y se añade un conteo de coberturas distintas. Esto permite distinguir una tarjeta que incluye un único seguro de otra con un paquete amplio de protección.

In [ ]:
CATEGORIAS_SEGURO = {
    "accidentes_viaje": r"accidentes? en viaje",
    "emergencia_medica": r"emergencia medica",
    "equipaje": r"equipaje",
    "cancelacion_demora_vuelo": r"cancelacion|demora de vuelo",
    "renta_auto": r"renta de auto"
}

CATEGORIAS_SEGURO_VIAJE = set(CATEGORIAS_SEGURO.keys())

seguros_modelo = construir_features_categorias(
    beneficios_fe, "Seguros", "benef_seguro", CATEGORIAS_SEGURO, CATEGORIAS_SEGURO_VIAJE
)

display(seguros_modelo.head())

### Anualidad y resumen global de beneficios

El beneficio “sin anualidad” es necesariamente binario, pero se incorpora junto con medidas globales de diversidad y cantidad de beneficios para evitar que el perfil quede representado únicamente por marcas de presencia.

In [ ]:
anualidad_benef = beneficios_fe[beneficios_fe["Tipo"].eq("Anualidad")].copy()
anualidad_benef["benef_sin_anualidad"] = anualidad_benef["texto_norm"].str.contains("sin anualidad").astype(int)

anualidad_modelo = anualidad_benef.groupby("ID_Tarjeta").agg(
    benef_anualidad_n_registros=("Tipo", "size"),
    benef_sin_anualidad=("benef_sin_anualidad", "max")
).reset_index()

resumen_beneficios = beneficios_fe.groupby("ID_Tarjeta").agg(
    benef_n_registros=("Tipo", "size"),
    benef_n_tipos=("Tipo", "nunique")
).reset_index()

conteos_tipo = pd.crosstab(beneficios_fe["ID_Tarjeta"], beneficios_fe["tipo_norm"])
conteos_tipo.columns = [f"benef_n_{c}" for c in conteos_tipo.columns]
conteos_tipo = conteos_tipo.reset_index()

display(resumen_beneficios.head())

In [ ]:
# Integración de todas las familias de beneficios.
beneficios_modelo = resumen_beneficios.merge(conteos_tipo, on="ID_Tarjeta", how="left")

for bloque in [
    puntos_modelo,
    transfer_modelo,
    msi_modelo,
    descuentos_modelo,
    preventas_modelo,
    seguros_modelo,
    anualidad_modelo
]:
    beneficios_modelo = beneficios_modelo.merge(bloque, on="ID_Tarjeta", how="left")

# Ausencia de una fila de beneficio = ausencia de ese beneficio, por eso se codifica con 0.
beneficios_modelo = beneficios_modelo.fillna(0)

# Indicadores de amplitud total construidos a partir de las variables ya generadas.
cols_categorias = [c for c in beneficios_modelo.columns if c.endswith("_n_categorias")]
cols_viaje = [c for c in beneficios_modelo.columns if c.endswith("_n_categorias_viaje")]
cols_consumo = [c for c in beneficios_modelo.columns if c.endswith("_n_categorias_consumo")]

beneficios_modelo["benef_amplitud_categorias_total"] = beneficios_modelo[cols_categorias].sum(axis=1)
beneficios_modelo["benef_amplitud_viaje"] = beneficios_modelo[cols_viaje].sum(axis=1)
beneficios_modelo["benef_amplitud_consumo"] = beneficios_modelo[cols_consumo].sum(axis=1)

print("Dimensión de variables de beneficios:", beneficios_modelo.shape)
display(beneficios_modelo.head())

## Conjuntar

En este punto cada tabla transformada debe tener una fila por `ID_Tarjeta`. Se integran las variables en un dataset maestro. Los campos de identificación y perfilamiento permanecen junto a las variables del modelo para facilitar la interpretación posterior.

In [ ]:
for nombre, df in {
    "tarjetas_fe": tarjetas_fe,
    "requisitos_modelo": requisitos_modelo,
    "comisiones_modelo": comisiones_modelo,
    "beneficios_modelo": beneficios_modelo
}.items():
    print(nombre, df.shape, "IDs únicos:", df["ID_Tarjeta"].nunique())


datos = (
    tarjetas_fe
    .merge(requisitos_modelo, on="ID_Tarjeta", how="left", validate="one_to_one")
    .merge(comisiones_modelo, on="ID_Tarjeta", how="left", validate="one_to_one")
    .merge(beneficios_modelo, on="ID_Tarjeta", how="left", validate="one_to_one")
)

# Para beneficios/comisiones, ausencia de registro implica ausencia de la característica.
cols_estructurales = [c for c in datos.columns if c.startswith("com_") or c.startswith("benef_")]
datos[cols_estructurales] = datos[cols_estructurales].fillna(0)

print("Dataset maestro:", datos.shape)
display(datos.head())

## Validaciones después de la ingeniería

Se revisa que no se hayan creado duplicados, que todas las tarjetas permanezcan en la base y que las variables destinadas al modelo no contengan faltantes inesperados.

In [ ]:
print("Filas:", len(datos))
print("IDs únicos:", datos["ID_Tarjeta"].nunique())
print("Duplicados por ID:", datos["ID_Tarjeta"].duplicated().sum())

display(tabla_completitud(datos).head(40))

## EDA del dataset diseñado

Una vez terminada la ingeniería de variables, se repite la exploración sobre el conjunto que realmente utilizará el modelo. En esta etapa interesa detectar escalas muy diferentes, variables constantes, variables casi idénticas y correlaciones fuertes.

La correlación no implica que una variable deba eliminarse automáticamente; sirve para identificar redundancias que conviene revisar antes de la versión final.

In [ ]:
variables_numericas_disponibles = datos.select_dtypes(include=np.number).columns.tolist()

# Variables administrativas/originales que no se analizarán como features finales.
no_modelo_base = [
    "conteo_requisitos_original",
    "conteo_comisiones_original",
    "conteo_beneficios_original"
]

variables_eda = [c for c in variables_numericas_disponibles if c not in no_modelo_base]
print("Variables numéricas diseñadas:", len(variables_eda))
display(desc_table(datos, variables_eda, "num"))

In [ ]:
# Variables constantes: no aportan separación a ningún algoritmo de clustering.
variaciones = datos[variables_eda].nunique(dropna=False).sort_values()
variables_constantes = variaciones[variaciones <= 1].index.tolist()
print("Variables constantes:", variables_constantes)

# Pares con correlación absoluta alta.
pares_altos = pares_correlacion_alta(datos[variables_eda], umbral=0.90)
display(pares_altos.head(50))

## Revisión visual de posibles valores extremos

Se utilizan histogramas y boxplots para entender la forma de las variables. **No se eliminarán observaciones por ser atípicas.** El objetivo es reconocer si existen tarjetas que representan segmentos de costo, línea o beneficios particularmente altos o bajos.

In [ ]:
vars_financieras_clave = [
    "cat_pct", "anualidad_pesos", "tasa_interes_pct", "linea_credito_desde",
    "ingreso_mensual_minimo_modelo", "score_minimo_modelo",
    "benef_n_registros", "benef_n_tipos", "benef_amplitud_categorias_total"
]
vars_financieras_clave = [c for c in vars_financieras_clave if c in datos.columns]

graficar_numericas(datos, vars_financieras_clave, ncols=3)

fig, axes = plt.subplots(
    int(np.ceil(len(vars_financieras_clave)/3)), 3,
    figsize=(15, 3.2*int(np.ceil(len(vars_financieras_clave)/3)))
)
axes = np.array(axes).reshape(-1)
for ax, col in zip(axes, vars_financieras_clave):
    sb.boxplot(x=datos[col], ax=ax)
    ax.set_title(col)
for ax in axes[len(vars_financieras_clave):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Definición de variables para clustering

Por defecto se excluyen del modelo:

- `ID_Tarjeta` y nombre: identificadores.
- `institucion`: podría hacer que el algoritmo agrupe por banco en lugar de por propuesta de producto.
- `clase`: es una etiqueta comercial ya existente y puede condicionar el resultado que se pretende descubrir.
- `duplicado` y los conteos originales: campos administrativos o redundantes con la ingeniería realizada.

Se mantienen dos interruptores para probar `Clase` o `Institución` posteriormente, utilizando el mismo One-Hot Encoding que ya se ha trabajado anteriormente.

In [ ]:
USAR_CLASE_EN_MODELO = False
USAR_INSTITUCION_EN_MODELO = False

variables_identificacion = ["ID_Tarjeta", "nombre_tarjeta"]
variables_perfil = ["institucion", "clase"]
variables_administrativas = [
    "duplicado",
    "conteo_requisitos_original",
    "conteo_comisiones_original",
    "conteo_beneficios_original"
]

# No se utilizan las columnas originales con NaN de requisitos; se usan sus versiones estructurales.
variables_req_originales_no_modelo = []

variables_categoricas_modelo = []
if USAR_CLASE_EN_MODELO:
    variables_categoricas_modelo.append("clase")
if USAR_INSTITUCION_EN_MODELO:
    variables_categoricas_modelo.append("institucion")

excluir = set(
    variables_identificacion + variables_perfil + variables_administrativas +
    variables_req_originales_no_modelo + variables_constantes
)

variables_numericas_modelo = [
    c for c in datos.select_dtypes(include=np.number).columns
    if c not in excluir
]

print("Numéricas para el modelo:", len(variables_numericas_modelo))
print("Categóricas para el modelo:", variables_categoricas_modelo)

## Ajuste manual opcional después del EDA

La siguiente lista queda intencionalmente vacía en esta primera versión. Después de ejecutar el notebook se puede utilizar la tabla de correlaciones y el perfilamiento para decidir si alguna variable es redundante o conceptualmente inadecuada.

No se elimina automáticamente una variable únicamente por estar correlacionada: primero debe revisarse si aporta una interpretación distinta.

In [ ]:
VARIABLES_A_EXCLUIR_MANUALMENTE = [
    # Ejemplo: "alguna_variable_redundante"
]

variables_numericas_modelo = [
    c for c in variables_numericas_modelo
    if c not in VARIABLES_A_EXCLUIR_MANUALMENTE
]

# Transformación y escalamiento

K-Means y el método de Ward trabajan con distancias. Por ello es necesario estandarizar las variables numéricas para evitar que una variable medida en pesos domine a otra medida en porcentajes o conteos.

No se utiliza imputación. Si queda algún `NaN` en las variables del modelo, la celda detendrá el flujo para obligar a revisar su origen.

In [ ]:
X_num = datos[variables_numericas_modelo].copy()

if X_num.isna().any().any():
    faltantes_modelo = X_num.columns[X_num.isna().any()].tolist()
    raise ValueError(f"Hay NaN inesperados en variables del modelo: {faltantes_modelo}")

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

# One-Hot Encoding opcional para Clase/Institución.
encoder = None
if variables_categoricas_modelo:
    encoder = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
    X_cat = encoder.fit_transform(datos[variables_categoricas_modelo])
    nombres_cat = encoder.get_feature_names_out(variables_categoricas_modelo).tolist()
    X = np.hstack([X_num_scaled, X_cat])
    nombres_features = variables_numericas_modelo + nombres_cat
else:
    X = X_num_scaled
    nombres_features = variables_numericas_modelo.copy()

print("Matriz final para clustering:", X.shape)
print("Número de variables:", len(nombres_features))

# Reducción de dimensionalidad

## Varianza explicada total

Primero se ajusta PCA con todos los componentes posibles para observar la varianza explicada acumulada. Después se determina automáticamente el número mínimo de componentes que alcanza al menos **90%**.

In [ ]:
pca_total = PCA()
X_pca_total = pca_total.fit_transform(X)

varianza = pca_total.explained_variance_ratio_
varianza_acum = np.cumsum(varianza)

n_componentes_90 = np.argmax(varianza_acum >= 0.90) + 1
print("Componentes para alcanzar al menos 90%:", n_componentes_90)
print("Varianza acumulada:", varianza_acum[n_componentes_90 - 1])

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(varianza_acum)+1), varianza_acum, marker="o")
plt.axhline(0.90, linestyle="--")
plt.axvline(n_componentes_90, linestyle="--")
plt.xlabel("Número de componentes")
plt.ylabel("Varianza explicada acumulada")
plt.title("PCA - Varianza explicada acumulada")
plt.show()

## PCA para clusterizar

Se crea `X_pca90`, que conserva el número de componentes necesario para explicar aproximadamente el 90% de la varianza. Esta será la segunda representación de los datos que se comparará contra las variables escaladas originales.

In [ ]:
pca90 = PCA(n_components=n_componentes_90)
X_pca90 = pca90.fit_transform(X)

print("Dimensión original:", X.shape)
print("Dimensión PCA 90%:", X_pca90.shape)
print("Varianza total conservada:", pca90.explained_variance_ratio_.sum())

## PCA de 2 y 3 componentes para visualización

Estos PCA no se utilizarán necesariamente para entrenar el modelo final. Su objetivo es proyectar las tarjetas a dos y tres dimensiones para visualizar la separación entre clusters.

In [ ]:
pca2 = PCA(n_components=2)
X_pca2 = pca2.fit_transform(X)

pca3 = PCA(n_components=3)
X_pca3 = pca3.fit_transform(X)

print("Varianza explicada con 2 componentes:", pca2.explained_variance_ratio_.sum())
print("Varianza explicada con 3 componentes:", pca3.explained_variance_ratio_.sum())

# Modelado de clustering y experimentos de dimensionalidad

Se evaluarán los mismos datos bajo dos representaciones:

- **Original/escalada (`X`)**: conserva todas las variables seleccionadas.
- **PCA 90% (`X_pca90`)**: reduce dimensionalidad manteniendo la mayor parte de la información.

Para K-Means se utilizan codo y silueta. Para aglomerativo se utiliza dendrograma y silueta. La silueta permite comparar escenarios en una escala común, pero la decisión final también debe considerar el tamaño de los grupos y su interpretación.

## Experimento 1: K-Means sin PCA

In [ ]:
resultados_kmeans_original = evaluar_kmeans(X, k_min=2, k_max=10)
display(resultados_kmeans_original)
graficar_codo_silueta(resultados_kmeans_original, "K-Means sin PCA")

k_kmeans_original = int(
    resultados_kmeans_original.loc[resultados_kmeans_original["silhouette"].idxmax(), "k"]
)
print("k sugerido por silueta:", k_kmeans_original)

modelo_kmeans_original = KMeans(
    n_clusters=k_kmeans_original,
    random_state=RANDOM_STATE,
    n_init=20
)
labels_kmeans_original = modelo_kmeans_original.fit_predict(X)
grafica_silueta(X, labels_kmeans_original, "K-Means sin PCA")

## Experimento 2: K-Means con PCA al 90%

In [ ]:
resultados_kmeans_pca = evaluar_kmeans(X_pca90, k_min=2, k_max=10)
display(resultados_kmeans_pca)
graficar_codo_silueta(resultados_kmeans_pca, "K-Means con PCA 90%")

k_kmeans_pca = int(
    resultados_kmeans_pca.loc[resultados_kmeans_pca["silhouette"].idxmax(), "k"]
)
print("k sugerido por silueta:", k_kmeans_pca)

modelo_kmeans_pca = KMeans(
    n_clusters=k_kmeans_pca,
    random_state=RANDOM_STATE,
    n_init=20
)
labels_kmeans_pca = modelo_kmeans_pca.fit_predict(X_pca90)
grafica_silueta(X_pca90, labels_kmeans_pca, "K-Means con PCA 90%")

## Experimento 3: Clustering aglomerativo sin PCA

Se utiliza el método de **Ward**, apropiado para variables numéricas estandarizadas y distancia euclidiana. El dendrograma ayuda a observar dónde se producen saltos importantes en distancia; después se calcula silueta para varios valores de `k`.

In [ ]:
Z_original = graficar_dendrograma(
    X,
    titulo="Dendrograma - Aglomerativo sin PCA",
    metodo="ward"
)

resultados_aglo_original = evaluar_aglomerativo(X, k_min=2, k_max=10, linkage_method="ward")
display(resultados_aglo_original)
graficar_silueta_aglomerativo(resultados_aglo_original, "Aglomerativo sin PCA")

k_aglo_original = int(
    resultados_aglo_original.loc[resultados_aglo_original["silhouette"].idxmax(), "k"]
)
print("k sugerido por silueta:", k_aglo_original)

modelo_aglo_original = AgglomerativeClustering(
    n_clusters=k_aglo_original,
    linkage="ward"
)
labels_aglo_original = modelo_aglo_original.fit_predict(X)
grafica_silueta(X, labels_aglo_original, "Aglomerativo sin PCA")

## Experimento 4: Clustering aglomerativo con PCA al 90%

In [ ]:
Z_pca = graficar_dendrograma(
    X_pca90,
    titulo="Dendrograma - Aglomerativo con PCA 90%",
    metodo="ward"
)

resultados_aglo_pca = evaluar_aglomerativo(X_pca90, k_min=2, k_max=10, linkage_method="ward")
display(resultados_aglo_pca)
graficar_silueta_aglomerativo(resultados_aglo_pca, "Aglomerativo con PCA 90%")

k_aglo_pca = int(
    resultados_aglo_pca.loc[resultados_aglo_pca["silhouette"].idxmax(), "k"]
)
print("k sugerido por silueta:", k_aglo_pca)

modelo_aglo_pca = AgglomerativeClustering(
    n_clusters=k_aglo_pca,
    linkage="ward"
)
labels_aglo_pca = modelo_aglo_pca.fit_predict(X_pca90)
grafica_silueta(X_pca90, labels_aglo_pca, "Aglomerativo con PCA 90%")

# Comparación de escenarios

Se resume el mejor resultado de silueta obtenido dentro de cada escenario. Esta tabla **no sustituye** la revisión del codo, los dendrogramas, el tamaño de los clusters y su interpretabilidad; funciona como una guía cuantitativa común.

En esta primera versión el escenario con mayor silueta se selecciona automáticamente como propuesta. Después de revisar las gráficas se puede sobrescribir `ESCENARIO_FINAL` manualmente.

In [ ]:
comparacion = pd.DataFrame([
    {
        "Escenario": "KMeans_original",
        "k": k_kmeans_original,
        "Silhouette": resultados_kmeans_original["silhouette"].max()
    },
    {
        "Escenario": "KMeans_PCA90",
        "k": k_kmeans_pca,
        "Silhouette": resultados_kmeans_pca["silhouette"].max()
    },
    {
        "Escenario": "Aglomerativo_original",
        "k": k_aglo_original,
        "Silhouette": resultados_aglo_original["silhouette"].max()
    },
    {
        "Escenario": "Aglomerativo_PCA90",
        "k": k_aglo_pca,
        "Silhouette": resultados_aglo_pca["silhouette"].max()
    }
]).sort_values("Silhouette", ascending=False).reset_index(drop=True)

display(comparacion)

ESCENARIO_FINAL = comparacion.loc[0, "Escenario"]
print("Escenario propuesto por mayor silueta:", ESCENARIO_FINAL)

# Si la revisión visual/metodológica sugiere otro, escribir aquí exactamente uno de los cuatro nombres.
# ESCENARIO_FINAL = "KMeans_original"
# ESCENARIO_FINAL = "KMeans_PCA90"
# ESCENARIO_FINAL = "Aglomerativo_original"
# ESCENARIO_FINAL = "Aglomerativo_PCA90"

In [ ]:
if ESCENARIO_FINAL == "KMeans_original":
    labels_final = labels_kmeans_original
    modelo_final = modelo_kmeans_original
    X_final = X
elif ESCENARIO_FINAL == "KMeans_PCA90":
    labels_final = labels_kmeans_pca
    modelo_final = modelo_kmeans_pca
    X_final = X_pca90
elif ESCENARIO_FINAL == "Aglomerativo_original":
    labels_final = labels_aglo_original
    modelo_final = modelo_aglo_original
    X_final = X
elif ESCENARIO_FINAL == "Aglomerativo_PCA90":
    labels_final = labels_aglo_pca
    modelo_final = modelo_aglo_pca
    X_final = X_pca90
else:
    raise ValueError("ESCENARIO_FINAL no reconocido")

print("Silueta del escenario final:", silhouette_score(X_final, labels_final))
print("Tamaño de clusters:")
print(pd.Series(labels_final).value_counts().sort_index())

# Visualización con PCA

Los clusters finales se proyectan sobre PCA de 2 y 3 componentes. Estas gráficas permiten observar la separación aproximada, pero deben interpretarse junto con el porcentaje de varianza que retienen: una proyección de baja dimensión puede ocultar diferencias presentes en el espacio completo.

In [ ]:
visual_2d = pd.DataFrame({
    "PC1": X_pca2[:, 0],
    "PC2": X_pca2[:, 1],
    "Cluster": labels_final,
    "Tarjeta": datos["nombre_tarjeta"].values
})

plt.figure(figsize=(10, 7))
sb.scatterplot(data=visual_2d, x="PC1", y="PC2", hue="Cluster", palette="tab10", s=90)
plt.title("Clusters de tarjetas - PCA 2D")
plt.legend(title="Cluster")
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(
    X_pca3[:, 0],
    X_pca3[:, 1],
    X_pca3[:, 2],
    c=labels_final,
    cmap="tab10",
    s=65
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("Clusters de tarjetas - PCA 3D")
fig.colorbar(scatter, ax=ax, label="Cluster")
plt.show()

# Perfilamiento

El clustering se ejecutó con variables transformadas y escaladas, pero la interpretación debe regresar a las **variables originales y derivadas en unidades comprensibles**. El objetivo es describir qué distingue a cada grupo sin asignar etiquetas subjetivas antes de observar los resultados.

In [ ]:
datos_cluster = datos.copy()
datos_cluster["Cluster"] = labels_final

display(
    datos_cluster[
        ["ID_Tarjeta", "nombre_tarjeta", "institucion", "clase", "Cluster"]
    ].sort_values(["Cluster", "institucion", "nombre_tarjeta"])
)

## Perfil numérico por cluster

Se calculan medias y medianas de las variables utilizadas para modelar. Debido al número potencialmente grande de características, posteriormente se ordenarán por capacidad de separación entre clusters.

In [ ]:
perfil_media = datos_cluster.groupby("Cluster")[variables_numericas_modelo].mean().T
perfil_mediana = datos_cluster.groupby("Cluster")[variables_numericas_modelo].median().T

print("Perfil por medias:")
display(perfil_media)

print("Perfil por medianas:")
display(perfil_mediana)

## Variables que más distinguen los clusters

Siguiendo la lógica del ejercicio previo, se calcula la diferencia entre el promedio máximo y mínimo de cada variable entre clusters. Para que escalas distintas sean comparables, esta comparación se realiza también sobre las variables estandarizadas.

In [ ]:
# Perfil estandarizado únicamente para comparar magnitudes entre variables.
df_scaled = pd.DataFrame(X_num_scaled, columns=variables_numericas_modelo, index=datos.index)
df_scaled["Cluster"] = labels_final

promedios_z = df_scaled.groupby("Cluster").mean()
diferencias_z = (promedios_z.max() - promedios_z.min()).sort_values(ascending=False)

print("Variables con mayor separación entre clusters (rango de medias estandarizadas):")
display(diferencias_z.head(30).to_frame("rango_medias_z"))

In [ ]:
TOP_N = min(20, len(diferencias_z))
top_variables = diferencias_z.head(TOP_N).index.tolist()

plt.figure(figsize=(12, max(6, TOP_N * 0.35)))
sb.heatmap(
    promedios_z[top_variables].T,
    cmap="vlag",
    center=0,
    annot=True,
    fmt=".2f"
)
plt.title("Perfil estandarizado de las variables que más separan los clusters")
plt.xlabel("Cluster")
plt.ylabel("Variable")
plt.show()

## Perfil comercial: institución y clase

Aunque `Institución` y `Clase` no entren por defecto al clustering, pueden ayudar a interpretar la composición de los grupos. Se muestran como variables externas de perfilamiento y no como causas directas de la agrupación.

In [ ]:
print("Distribución de institución dentro de cada cluster (%):")
display(
    pd.crosstab(datos_cluster["Cluster"], datos_cluster["institucion"], normalize="index") * 100
)

print("Distribución de clase dentro de cada cluster (%):")
display(
    pd.crosstab(datos_cluster["Cluster"], datos_cluster["clase"], normalize="index") * 100
)

## Resumen de variables clave

Esta vista reduce el perfil a variables financieras, requisitos y medidas agregadas de beneficios. Sirve como primera base para redactar una descripción cualitativa de cada segmento una vez observados los resultados reales.

In [ ]:
variables_resumen = [
    "cat_pct",
    "anualidad_pesos",
    "tasa_interes_pct",
    "linea_credito_desde",
    "edad_minima",
    "score_minimo_modelo",
    "ingreso_mensual_minimo_modelo",
    "com_n_conceptos",
    "benef_n_registros",
    "benef_n_tipos",
    "benef_amplitud_categorias_total",
    "benef_amplitud_viaje",
    "benef_amplitud_consumo",
    "benef_puntos_por_peso_max",
    "benef_puntos_por_dolar_max",
    "benef_transfer_tasa_min",
    "benef_transfer_plazo_max"
]
variables_resumen = [c for c in variables_resumen if c in datos_cluster.columns]

display(datos_cluster.groupby("Cluster")[variables_resumen].mean().T)

# Exportación de resultados

Se guardan dos archivos para continuar el proyecto:

- `dataset_tarjetas_ingenieria.csv`: dataset maestro después de la ingeniería de variables.
- `tarjetas_clusterizadas.csv`: identificación de cada tarjeta y cluster asignado.

La clasificación de un usuario nuevo y el recomendador de tarjetas se desarrollarán posteriormente. Es importante notar que K-Means dispone de `predict()` para nuevas observaciones, mientras que el clustering aglomerativo no asigna directamente nuevos casos; si este último resulta seleccionado, se definirá después una estrategia coherente para la etapa de recomendación.

In [ ]:
datos.to_csv("/content/dataset_tarjetas_ingenieria.csv", index=False, encoding="utf-8-sig")

tarjetas_clusterizadas = datos_cluster[
    ["ID_Tarjeta", "nombre_tarjeta", "institucion", "clase", "Cluster"]
].copy()
tarjetas_clusterizadas.to_csv(
    "/content/tarjetas_clusterizadas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos generados en /content/")

# Conclusiones pendientes de ejecución

Esta primera versión del notebook deja construido el pipeline metodológico, pero las conclusiones deben redactarse **después de ejecutar los datos reales**. En particular se deberá documentar:

1. Cuántas variables resultaron de la ingeniería y cuáles fueron constantes o muy redundantes.
2. Cuántos componentes fueron necesarios para conservar al menos 90% de la varianza.
3. Qué valores de `k` sugirieron codo, silueta y dendrograma.
4. Qué escenario produjo una estructura de clusters más clara e interpretable.
5. Qué variables financieras, requisitos, comisiones y beneficios distinguen a cada grupo.
6. Si los clusters encontrados son útiles como base para la futura etapa **perfil de usuario → segmento → recomendación de tarjeta**.

Después de la primera ejecución conviene revisar especialmente los regex de `Beneficios`, las variables altamente correlacionadas y el balance de tamaños entre clusters antes de cerrar el modelo definitivo.